In [7]:
import os
import json
from anthropic import Anthropic

client = Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY")
)

model = "claude-sonnet-5-5"


def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })


def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)

    return response.content[0].text


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation.

The dataset will be used to evaluate prompts that generate Python,
JSON, or Regex specifically for AWS-related tasks.

Return ONLY valid JSON in this format:

[
  {
    "task": "Description of task"
  }
]

Requirements:
- Generate exactly 3 objects.
- Focus on AWS-related tasks.
- Each task must be solvable using a single Python function,
  a single JSON object, or a single regex.
- Do not include markdown code fences.
- Do not include explanations.
"""

    messages = []

    add_user_message(messages, prompt)

    text = chat(messages)

    return json.loads(text)


dataset = generate_dataset()

print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a Python function named `list_untagged_ec2_instances` that takes a boto3 EC2 client and a required tag key (string), paginates through `describe_instances`, and returns a sorted list of instance IDs that do not have the specified tag key.'}, {'task': "Create a single JSON object representing an AWS IAM policy document (Version 2012-10-17) that allows only the s3:GetObject and s3:PutObject actions on objects in the bucket 'example-app-data' under the prefix 'uploads/', and explicitly denies all S3 actions on that bucket and its objects when the request is not made over HTTPS (aws:SecureTransport is false)."}, {'task': "Write a single regular expression that matches a valid AWS S3 ARN for an object, in the format 'arn:aws:s3:::bucket-name/key-path', where the bucket name is 3-63 characters long, consisting of lowercase letters, numbers, hyphens, and periods, starting and ending with a letter or number, and the key path is one or more characters after the first slash."}]
